In [56]:
# Libraries for HTTP requests and data manipulation
import requests
import pandas as pd 

# Environment variables (passwords, sensitive config)
import os 
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("NVD_API_KEY")

# Suppress warning messages to keep the output clean
import warnings
warnings.filterwarnings("ignore")

# Time management and rate limiting
import time
from datetime import datetime, timedelta

import csv

In [3]:
# NVD API base endpoint
url = "https://services.nvd.nist.gov/rest/json/cves/2.0"

# Query parameters for the request
params = {
    "resultsPerPage": 5
}

# Request headers including API key authentication
headers = {
    "apiKey": api_key
}

In [4]:
def api_requests(url, params, headers, max_retries=3, wait_seconds=5):
    
    """
    Executes an HTTP GET request to the specified API endpoint with retry logic 
    for temporary errors and network exceptions.

    Parameters:
        url (str): The target API endpoint.
        params (dict): Query parameters for the request.
        headers (dict): HTTP headers including authentication keys.
        max_retries (int): Maximum number of retry attempts for transient failures (default: 3).
        wait_seconds (int): Delay in seconds between retry attempts (default: 5).

    Returns:
        dict or None: Parsed JSON response if successful, None otherwise.
    """

    for attempt in range(1, max_retries + 1):
        try:
            # Send a GET request to the API URL 
            nvd_data = requests.get(url, params=params, headers=headers)

            if nvd_data.status_code == 200:
                print("API connected")
                # Convert the data to JSON
                return nvd_data.json()

            # Temporary errors: worth retrying
            elif nvd_data.status_code in [429, 500, 502, 503, 504]:
                print(f"API failed with {nvd_data.status_code} (temporary). Attempt {attempt}/{max_retries}")
                time.sleep(wait_seconds)
                continue

            # Structural errors: retrying won't help
            else:
                print(f"API failed with {nvd_data.status_code} (not retryable)")
                print(nvd_data.text)
                return None

        # Handle connection-related errors (e.g., DNS failure, refused connection)        
        except requests.exceptions.ConnectionError as CnxE:
            print(f"Connection error: {CnxE}. Attempt {attempt}/{max_retries}")
            time.sleep(wait_seconds)
            continue

        # Handle requests that exceed the timeout limit
        except requests.exceptions.Timeout as TO:
            print(f"Timeout: {TO}. Attempt {attempt}/{max_retries}")
            time.sleep(wait_seconds)
            continue

        # Handle any other ambiguous exception that occurs while handling a request
        except requests.exceptions.RequestException as e:
            print(f"Request error: {e}")
            return None

    print("Stopped: all retries failed")
    return None

In [5]:
def get_all_cves(url, params, headers):

    """
    Fetches all CVE records matching the given parameters by handling API pagination.

    Parameters:
        url (str): The NVD API endpoint URL.
        params (dict): Request query parameters (must include 'resultsPerPage').
        headers (dict): Request headers (including API key).

    Returns:
        list: A list containing all retrieved vulnerability items.
    """
    
    # List to accumulate CVEs from all pages
    all_cves = []

    # Start from the first result
    start_index = 0

    while True:
        # Update startIndex in the params dict before each request
        params["startIndex"] = start_index

        # Reuse the existing function to make the request
        response = api_requests(url, params, headers)

        # If the request failed, stop the loop
        if response is None:
            print("Stopped: request failed")
            break

        # Extract the list of vulnerabilities from this page
        cves_page = response["vulnerabilities"]

        # Add them to the general list
        all_cves.extend(cves_page)

        # Total number of results available for these filters
        total_results = response["totalResults"]

        print(f"Downloaded {len(all_cves)} of {total_results}")

        # Move the index forward by the page size
        start_index += params["resultsPerPage"]

        # Stop if there are no more pages left
        if start_index >= total_results:
            break

        # Pause to respect the rate limit
        time.sleep(1)

    return all_cves

In [6]:
def generate_date_windows(start_year, end_date=None, window_days=120):
    """Generates a list of (start, end) date tuples, each at most window_days long."""

    # Define the initial start date (January 1st of start_year)
    start = datetime(start_year, 1, 1)

    # Set the end limit to provided date or current timestamp
    end = end_date or datetime.now()
    
    windows = []
    current_start = start

    # Loop to build fixed-range time windows until reaching the end date
    while current_start < end:
        # Calculate window boundary ensuring it does not exceed the overall end date
        current_end = min(current_start + timedelta(days=window_days), end)
        windows.append((current_start, current_end))
        
        # Advance the start point for the next iteration
        current_start = current_end
    
    return windows

In [7]:
# Generate 120-day time windows starting from the year 2021
date_windows = generate_date_windows(2021)
all_cves_full = []

# Iterate through each date range window to fetch CVE data
for start, end in date_windows:
    # Set request parameters for the current time window
    params = {
        "resultsPerPage": 2000,
        "pubStartDate": start.strftime("%Y-%m-%dT00:00:00.000"),
        "pubEndDate": end.strftime("%Y-%m-%dT00:00:00.000")
    }

    # Retrieve all CVEs for the current window and accumulate them
    cves_window = get_all_cves(url, params, headers)
    all_cves_full.extend(cves_window)

    # Progress update per window
    print(f"Window {start.date()} - {end.date()}: {len(cves_window)} CVEs. Total so far: {len(all_cves_full)}")


API connected
Downloaded 2000 of 6960
API connected
Downloaded 4000 of 6960
API connected
Downloaded 6000 of 6960
API connected
Downloaded 6960 of 6960
Window 2021-01-01 - 2021-05-01: 6960 CVEs. Total so far: 6960
API connected
Downloaded 2000 of 7139
API connected
Downloaded 4000 of 7139
API connected
Downloaded 6000 of 7139
API connected
Downloaded 7139 of 7139
Window 2021-05-01 - 2021-08-29: 7139 CVEs. Total so far: 14099
API connected
Downloaded 2000 of 7650
API connected
Downloaded 4000 of 7650
API connected
Downloaded 6000 of 7650
API connected
Downloaded 7650 of 7650
Window 2021-08-29 - 2021-12-27: 7650 CVEs. Total so far: 21749
API connected
Downloaded 2000 of 8368
API connected
Downloaded 4000 of 8368
API connected
Downloaded 6000 of 8368
API connected
Downloaded 8000 of 8368
API connected
Downloaded 8368 of 8368
Window 2021-12-27 - 2022-04-26: 8368 CVEs. Total so far: 30117
API connected
Downloaded 2000 of 8483
API connected
Downloaded 4000 of 8483
API connected
Downloaded 60

In [8]:
def extract_cve_fields(cve_item):

    """
    Parses a single raw CVE dictionary from the NVD API and extracts a flat feature dictionary.

    Handles cross-version metrics fallback (CVSS v3.1 -> v3.0 -> v2.0), English description 
    filtering, primary CWE weakness extraction, and parses all associated CPE vendor/product pairs.

    Parameters:
        cve_item (dict): A raw vulnerability item from the NVD API response JSON.

    Returns:
        dict: A flattened dictionary containing structured CVE metadata and metrics.
    """

    cve = cve_item["cve"]

    # --- Basic fields ---
    cve_id = cve.get("id")
    published = cve.get("published")
    last_modified = cve.get("lastModified")
    vuln_status = cve.get("vulnStatus")

    # --- Description (English) ---
    description = None
    for desc in cve.get("descriptions", []):
        if desc.get("lang") == "en":
            description = desc.get("value")
            break

    # --- CVSS: try v3.1 and v3.0 first, then v2 ---
    metrics = cve.get("metrics", {})

    cvss_version = None
    base_score = None
    base_severity = None
    attack_vector = None
    attack_complexity = None
    privileges_required = None
    user_interaction = None
    confidentiality_impact = None
    integrity_impact = None
    availability_impact = None
    exploitability_score = None
    impact_score = None
    vector_string = None

    if "cvssMetricV31" in metrics or "cvssMetricV30" in metrics:
        key = "cvssMetricV31" if "cvssMetricV31" in metrics else "cvssMetricV30"
        metric = metrics[key][0]
        data = metric["cvssData"]

        cvss_version = "3.1" if key == "cvssMetricV31" else "3.0"
        base_score = data.get("baseScore")
        base_severity = data.get("baseSeverity")  # v3: inside cvssData
        attack_vector = data.get("attackVector")
        attack_complexity = data.get("attackComplexity")
        privileges_required = data.get("privilegesRequired")
        user_interaction = data.get("userInteraction")
        confidentiality_impact = data.get("confidentialityImpact")
        integrity_impact = data.get("integrityImpact")
        availability_impact = data.get("availabilityImpact")
        vector_string = data.get("vectorString")
        exploitability_score = metric.get("exploitabilityScore")
        impact_score = metric.get("impactScore")

    elif "cvssMetricV2" in metrics:
        metric = metrics["cvssMetricV2"][0]
        data = metric["cvssData"]

        cvss_version = "2.0"
        base_score = data.get("baseScore")
        base_severity = metric.get("baseSeverity")  # v2: OUTSIDE cvssData, at metric level
        attack_vector = data.get("accessVector")
        attack_complexity = data.get("accessComplexity")
        confidentiality_impact = data.get("confidentialityImpact")
        integrity_impact = data.get("integrityImpact")
        availability_impact = data.get("availabilityImpact")
        vector_string = data.get("vectorString")
        exploitability_score = metric.get("exploitabilityScore")
        impact_score = metric.get("impactScore")
        # privileges_required / user_interaction stay None: v2 doesn't have these concepts

    # --- CWE (first weakness found) ---
    cwe = None
    weaknesses = cve.get("weaknesses", [])
    if weaknesses:
        cwe = weaknesses[0]["description"][0].get("value")

    # --- Vendor / product: keep ALL cpeMatch entries as a list, not just the first ---
    vendors_products = []
    configurations = cve.get("configurations", [])
    if configurations:
        nodes = configurations[0].get("nodes", [])
        if nodes:
            cpe_matches = nodes[0].get("cpeMatch", [])
            for match in cpe_matches:
                criteria = match.get("criteria", "")
                parts = criteria.split(":")
                if len(parts) > 5:
                    vendors_products.append((parts[3], parts[4]))  # (vendor, product)

    # --- Number of references ---
    num_references = len(cve.get("references", []))

    return {
        "cve_id": cve_id,
        "published": published,
        "last_modified": last_modified,
        "vuln_status": vuln_status,
        "description": description,
        "cvss_version": cvss_version,
        "base_score": base_score,
        "base_severity": base_severity,
        "attack_vector": attack_vector,
        "attack_complexity": attack_complexity,
        "privileges_required": privileges_required,
        "user_interaction": user_interaction,
        "confidentiality_impact": confidentiality_impact,
        "integrity_impact": integrity_impact,
        "availability_impact": availability_impact,
        "exploitability_score": exploitability_score,
        "impact_score": impact_score,
        "vector_string": vector_string,
        "cwe": cwe,
        "vendors_products": vendors_products,
        "num_references": num_references
    }

In [9]:
# Flatten raw JSON records into structured dictionary objects
flat_cves = [extract_cve_fields(item) for item in all_cves_full]

# Convert the list of dictionaries into a Pandas DataFrame
df_cves = pd.DataFrame(flat_cves)

In [10]:
# Build the secondary table: one row per CVE-vendor-product combination
df_products = df_cves[["cve_id", "vendors_products"]].explode("vendors_products")

# Drop rows where the list was empty (explode turns [] into NaN)
df_products = df_products.dropna(subset=["vendors_products"])

# Split the (vendor, product) tuple into two separate columns
df_products[["vendor", "product"]] = pd.DataFrame(
    df_products["vendors_products"].tolist(), index=df_products.index
)

# Drop the now-unnecessary tuple column
df_products = df_products.drop(columns=["vendors_products"])

# Remove duplicate CVE-vendor-product rows (some CVEs list the same product multiple times, one per version)
df_products = df_products.drop_duplicates()

# Reset index for a clean final table
df_products = df_products.reset_index(drop=True)

In [11]:
# Drop the raw nested list column from the primary CVE DataFrame
df_cves = df_cves.drop(columns=["vendors_products"])

In [12]:
# Export clean DataFrames to CSV files without index columns
df_cves.to_csv("cves.csv", index=False)
df_products.to_csv("products.csv", index=False)

# EDA

## DF_CVES

In [13]:
df_cves.sample(5)

,cve_id,published,last_modified,vuln_status,description,cvss_version,base_score,base_severity,attack_vector,attack_complexity,privileges_required,user_interaction,confidentiality_impact,integrity_impact,availability_impact,exploitability_score,impact_score,vector_string,cwe,num_references
81507,CVE-2024-22729,2024-01-25T15:15:08.133,2026-06-17T07:11:42.850,Modified,NETIS SYSTEMS MW5360 V1.0.1.3031 was discovere...,3.1,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,5.9,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H,CWE-77,2
92680,CVE-2023-37331,2024-05-03T02:15:43.947,2026-06-17T06:08:00.423,Analyzed,Kofax Power PDF GIF File Parsing Stack-based B...,3.1,7.8,HIGH,LOCAL,LOW,NONE,REQUIRED,HIGH,HIGH,HIGH,1.8,5.9,CVSS:3.1/AV:L/AC:L/PR:N/UI:R/S:U/C:H/I:H/A:H,CWE-121,2
214865,CVE-2026-64257,2026-07-25T10:17:05.817,2026-07-30T15:00:27.343,Awaiting Analysis,"In the Linux kernel, the following vulnerabili...",3.1,9.1,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,NONE,HIGH,3.9,5.2,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:N/A:H,NaN,6
199827,CVE-2026-46244,2026-06-03T18:16:24.430,2026-07-22T20:10:00.127,Modified,"In the Linux kernel, the following vulnerabili...",3.1,9.1,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,NONE,3.9,5.2,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:N,NVD-CWE-noinfo,13
169253,CVE-2025-65570,2025-12-29T15:16:01.763,2026-06-17T09:55:48.683,Analyzed,A type confusion in jsish 2.0 allows incorrect...,3.1,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,5.9,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H,CWE-843,2


In [14]:
df_cves.info()

<class 'pandas.DataFrame'>
RangeIndex: 217827 entries, 0 to 217826
Data columns (total 20 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   cve_id                  217827 non-null  str    
 1   published               217827 non-null  str    
 2   last_modified           217827 non-null  str    
 3   vuln_status             217827 non-null  str    
 4   description             217827 non-null  str    
 5   cvss_version            202527 non-null  str    
 6   base_score              202527 non-null  float64
 7   base_severity           202527 non-null  str    
 8   attack_vector           202527 non-null  str    
 9   attack_complexity       202527 non-null  str    
 10  privileges_required     202517 non-null  str    
 11  user_interaction        202517 non-null  str    
 12  confidentiality_impact  202527 non-null  str    
 13  integrity_impact        202527 non-null  str    
 14  availability_impact     202527 

* _Check for duplicate records at both the DataFrame and unique ID levels._

In [15]:
df_cves.duplicated().sum()

np.int64(0)

In [16]:
df_cves['cve_id'].duplicated().sum()

np.int64(0)

* _Inspect unique values and data types in the `vuln_status` column._

In [17]:
df_cves['vuln_status'].value_counts()

vuln_status
Modified               97127
Analyzed               65215
Deferred               41730
Rejected                9271
Awaiting Analysis       2607
Received                1143
Undergoing Analysis      734
Name: count, dtype: int64

* _Verify that CVSS metrics across versions 2.0, 3.0, and 3.1 share consistent formatting and categories, accounting for the different extraction logic required per version._

In [18]:
df_cves['cvss_version'].value_counts()
      

cvss_version
3.1    200572
3.0      1945
2.0        10
Name: count, dtype: int64

In [19]:
df_cves['base_severity'].value_counts()

base_severity
MEDIUM      94021
HIGH        77328
CRITICAL    22738
LOW          8397
NONE           43
Name: count, dtype: int64

In [20]:
df_v3 = df_cves[df_cves['cvss_version'] == 3.0]
df_v3['base_severity'].value_counts()

Series([], Name: count, dtype: int64)

In [21]:
df_v2 = df_cves[df_cves['cvss_version'] == 2.0]
df_v2['base_severity'].value_counts()

Series([], Name: count, dtype: int64)

* _Verify that all `base_score` values fall within the valid 0 to 10 range_

In [22]:
((df_cves['base_score'] < 0) | (df_cves['base_score'] > 10)).any()

np.False_

* _Check for negative values across all numerical columns._

In [23]:
cves_numerical_cols = df_cves.select_dtypes(include='number')
(cves_numerical_cols < 0).any().any()

np.False_

* _Calculate the percentage of missing values for all incomplete columns._

In [24]:
cves_null_pct = round(df_cves.isnull().sum()/df_cves.shape[0]*100, 2)
cves_null_pct  = cves_null_pct [cves_null_pct >0]
cves_null_pct 

cvss_version              7.02
base_score                7.02
base_severity             7.02
attack_vector             7.02
attack_complexity         7.02
privileges_required       7.03
user_interaction          7.03
confidentiality_impact    7.02
integrity_impact          7.02
availability_impact       7.02
exploitability_score      7.02
impact_score              7.02
vector_string             7.02
cwe                       5.91
dtype: float64

In [25]:
df_base_null_rows = df_cves[df_cves["base_score"].isnull()]
df_base_null_rows.shape

(15300, 20)

In [26]:
# Confirm that missing values consistently occur within the same rows
cvss_cols = ["base_score", "base_severity", "attack_vector", "attack_complexity",
             "confidentiality_impact", "integrity_impact", "availability_impact",
             "exploitability_score", "impact_score", "vector_string"]

(df_base_null_rows[cvss_cols].isnull().sum(axis=1) == len(cvss_cols)).value_counts()

True    15300
Name: count, dtype: int64

In [27]:
# Examine vuln_status values across rows containing missing data
df_base_null_rows["vuln_status"].unique()

<StringArray>
[           'Rejected',            'Modified',            'Deferred',
   'Awaiting Analysis', 'Undergoing Analysis',            'Analyzed',
            'Received']
Length: 7, dtype: str

In [28]:
## Check how many records with missing values belong to the Modified or Analyzed statuses, 
# as these states imply that feature extraction should already be complete.
df_modified_analyzed_without_score = df_cves[
    (df_cves["base_score"].isnull()) &
    (df_cves["vuln_status"].isin(["Modified", "Analyzed"]))
]

df_modified_analyzed_without_score

,cve_id,published,last_modified,vuln_status,description,cvss_version,base_score,base_severity,attack_vector,attack_complexity,privileges_required,user_interaction,confidentiality_impact,integrity_impact,availability_impact,exploitability_score,impact_score,vector_string,cwe,num_references
41649,CVE-2022-32171,2022-10-06T18:16:02.610,2026-06-17T04:46:48.560,Modified,"In Zinc, versions v0.1.9 through v0.3.1 are vu...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CWE-79,4
41650,CVE-2022-32172,2022-10-06T18:16:03.630,2026-06-17T04:46:48.660,Modified,"In Zinc, versions v0.1.9 through v0.3.1 are vu...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CWE-79,4
79240,CVE-2023-4674,2023-12-29T15:15:09.497,2026-06-17T06:38:20.687,Modified,Improper Neutralization of Special Elements us...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CWE-89,3
132236,CVE-2025-2794,2025-03-31T17:15:41.333,2026-06-17T09:07:37.903,Modified,An unsafe reflection vulnerability in Kentico ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CWE-470,2
142798,CVE-2025-6151,2025-06-17T01:15:23.313,2026-06-17T10:01:16.420,Modified,A vulnerability has been found in \nTP-Link TL...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CWE-119,6
203487,CVE-2026-0092,2026-06-17T13:19:26.813,2026-06-18T04:16:33.567,Analyzed,"In Package Manager, there is a possible device...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CWE-862,1


In [29]:
sorted(df_cves['cwe'].dropna().unique(), reverse=True)

['NVD-CWE-noinfo',
 'NVD-CWE-Other',
 'CWE-99',
 'CWE-98',
 'CWE-97',
 'CWE-962',
 'CWE-96',
 'CWE-95',
 'CWE-943',
 'CWE-942',
 'CWE-941',
 'CWE-940',
 'CWE-94',
 'CWE-939',
 'CWE-93',
 'CWE-927',
 'CWE-926',
 'CWE-925',
 'CWE-924',
 'CWE-923',
 'CWE-922',
 'CWE-921',
 'CWE-920',
 'CWE-92',
 'CWE-918',
 'CWE-917',
 'CWE-916',
 'CWE-915',
 'CWE-914',
 'CWE-913',
 'CWE-912',
 'CWE-911',
 'CWE-910',
 'CWE-91',
 'CWE-909',
 'CWE-908',
 'CWE-90',
 'CWE-89',
 'CWE-882',
 'CWE-88',
 'CWE-87',
 'CWE-863',
 'CWE-862',
 'CWE-86',
 'CWE-85',
 'CWE-843',
 'CWE-842',
 'CWE-841',
 'CWE-840',
 'CWE-84',
 'CWE-839',
 'CWE-838',
 'CWE-837',
 'CWE-836',
 'CWE-835',
 'CWE-834',
 'CWE-833',
 'CWE-832',
 'CWE-830',
 'CWE-83',
 'CWE-829',
 'CWE-828',
 'CWE-826',
 'CWE-825',
 'CWE-824',
 'CWE-823',
 'CWE-822',
 'CWE-821',
 'CWE-820',
 'CWE-82',
 'CWE-815',
 'CWE-81',
 'CWE-807',
 'CWE-805',
 'CWE-804',
 'CWE-80',
 'CWE-799',
 'CWE-798',
 'CWE-794',
 'CWE-792',
 'CWE-791',
 'CWE-790',
 'CWE-79',
 'CWE-789',


In [30]:
null_rows = df_cves[
    df_cves["base_score"].isnull() &
    df_cves["cwe"].isnull()
]
null_rows["vuln_status"].unique()

<StringArray>
['Rejected', 'Deferred', 'Awaiting Analysis', 'Received']
Length: 4, dtype: str

In [31]:
df_rejected_with_data = df_cves[
    (df_cves["vuln_status"] == "Rejected") &
    (df_cves["base_score"].notnull())
]

df_rejected_with_data.shape

(0, 20)

In [32]:
df_deferred_with_data = df_cves[
    (df_cves["vuln_status"] == "Deferred") &
    (df_cves["base_score"].notnull())
]

df_deferred_with_data.shape

(36861, 20)

In [33]:
df_not_null = df_cves[df_cves["base_score"].notnull()]
df_not_null["vuln_status"].unique()

<StringArray>
[           'Modified',            'Analyzed',            'Deferred',
 'Undergoing Analysis',   'Awaiting Analysis',            'Received']
Length: 6, dtype: str

In [34]:
df_cves = df_cves[df_cves['vuln_status'] != 'Rejected']

In [35]:
df_cves['cwe_num'] = df_cves['cwe'].str.extract(r'CWE-(\d+)')
df_cves['cwe_num'] = df_cves['cwe_num'].astype('Int64')

In [36]:
df_cves['published'] = pd.to_datetime(df_cves['published'])

In [37]:
df_cves['last_modified'] = pd.to_datetime(df_cves['last_modified'])

In [38]:
df_cves['cvss_version'] = pd.to_numeric(df_cves['cvss_version'])

In [39]:
df_cves.info()

<class 'pandas.DataFrame'>
Index: 208556 entries, 0 to 217826
Data columns (total 21 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   cve_id                  208556 non-null  str           
 1   published               208556 non-null  datetime64[us]
 2   last_modified           208556 non-null  datetime64[us]
 3   vuln_status             208556 non-null  str           
 4   description             208556 non-null  str           
 5   cvss_version            202527 non-null  float64       
 6   base_score              202527 non-null  float64       
 7   base_severity           202527 non-null  str           
 8   attack_vector           202527 non-null  str           
 9   attack_complexity       202527 non-null  str           
 10  privileges_required     202517 non-null  str           
 11  user_interaction        202517 non-null  str           
 12  confidentiality_impact  202527 non-null  str  

In [40]:
cves_null_pct2 = round(df_cves.isnull().sum()/df_cves.shape[0]*100, 2)
cves_null_pct2  = cves_null_pct2 [cves_null_pct2 >0]
cves_null_pct2 

cvss_version              2.89
base_score                2.89
base_severity             2.89
attack_vector             2.89
attack_complexity         2.89
privileges_required       2.90
user_interaction          2.90
confidentiality_impact    2.89
integrity_impact          2.89
availability_impact       2.89
exploitability_score      2.89
impact_score              2.89
vector_string             2.89
cwe                       1.73
cwe_num                   9.05
dtype: float64

In [51]:
# Convert to string, replace '.' with ',', and remove leading/trailing spaces
for col in ['base_score', 'cvss_version', 'exploitability_score', 'impact_score']:
    df_cves[col] = df_cves[col].astype(str).str.strip().str.replace(".", ",", regex=False)

In [ ]:
# Remove internal line breaks (carriage returns and newlines) from description
df_cves["description"] = (
    df_cves["description"]
    .astype(str)
    .str.replace(r"[\r\n]+", " ", regex=True)
    .str.strip()
)

# Replace internal double quotes with single quotes to prevent CSV column splitting
df_cves["description"] = df_cves["description"].str.replace('"', "'")

# Escape leading formula characters (=, +, -, @) to prevent Power BI errors
df_cves["description"] = df_cves["description"].apply(
    lambda x: f"'{x}" if x.startswith(("=", "+", "-", "@")) else x
)


df_products

In [41]:
df_products.sample(5)

,cve_id,vendor,product
144132,CVE-2021-47241,linux,linux_kernel
148801,CVE-2024-39599,sap,sap_basis
5016,CVE-2020-11144,qualcomm,sd768g
122540,CVE-2023-41140,autodesk,autocad_map_3d
233162,CVE-2026-8201,mongodb,mongodb


In [42]:
df_products.info()

<class 'pandas.DataFrame'>
RangeIndex: 250364 entries, 0 to 250363
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   cve_id   250364 non-null  str  
 1   vendor   250364 non-null  str  
 2   product  250364 non-null  str  
dtypes: str(3)
memory usage: 5.7 MB


In [43]:
df_products.duplicated().sum()

np.int64(0)

In [44]:
# comoprobar q estan todos
df_products = df_products[df_products['cve_id'].isin(df_cves['cve_id'])]

In [45]:
df_products['vendor'].value_counts().head(10)

vendor
microsoft    44809
qualcomm     14217
linux        11934
intel        11424
apple         9559
google        8611
adobe         4608
f5            3837
ibm           3745
oracle        3628
Name: count, dtype: int64

In [46]:
df_products['product'].value_counts().head(10)

product
linux_kernel           11919
android                 5052
windows_server_2019     3505
chrome                  3499
windows_server_2016     3156
windows_server_2022     3125
macos                   2713
windows_server_2012     2510
windows_10_21h2         2334
windows_10_22h2         2332
Name: count, dtype: int64

In [ ]:
# Export to CSV using strict quotes (QUOTE_ALL) and UTF-8-SIG encoding for full compatibility
df_cves.to_csv(
    "cves_cleaned.csv",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
)

In [ ]:
df_products.to_csv("products_cleaned.csv", index=False)